# Avatar pipeline, offline — image + audio → talking-head video, **head frozen**

One reference photo and one audio file in, one lip-synced MP4 out, using **this repo's
production avatar pipeline** — the same generator, renderer, checkpoints and streaming
sample loop that `run_x86.sh` serves — with PersonaPlex replaced by an audio file.

This is not upstream IMTalker. Upstream moves the head while it talks; this pipeline does
not, and the difference is reproduced here rather than approximated. Section 12 explains
exactly which knob holds the head still and how to turn it off to see the difference.

| | |
|---|---|
| generator | `experiments.original_pod_8998.FM.FMGenerator` + your `generator_last.ckpt` |
| renderer | `renderer.models.IMTRenderer` + your `renderer.ckpt`, fp32 |
| audio front end | Wav2Vec2 (`wav2vec2-base-960h`) in place of Helium + `adapter_last.pt` |
| sample loop | `wav2vec_sec 2.0` window, `fm_chunk_frames 2` emitted per call, `stream_state` carried, shared noise seeded 42 |
| sampler | `a_cfg_scale 1.24`, `nfe 3`, `motion_ref_blend 0.0` — the `run_x86.sh` values |
| image | `src/imtalker/image/rb.jpeg` |
| audio | `src/imtalker/audio/rb_wav.wav` |

### Checkpoints — nothing of yours is required

`prepare_x86.sh` does not download the avatar checkpoints; on a prepared pod they are in
`weights/`. On a fresh pod they are not, so section 7 resolves everything from public
repos, and the generator it uses is the one named for exactly this behaviour:

| file | where section 7 gets it |
|---|---|
| generator | `niloy629/imtalker_lora` → **`lora_exp4_lookahead_120ms_static_head_last.ckpt`**. Its Lightning `hyper_parameters` say `exp_name = lora_exp4_lookahead_120ms_static_head`, `lora_rank 16`, `lora_alpha 32`, and its `base_ckpt` is `exps/exp4_lookahead_120ms/checkpoints/last.ckpt` — your `generator_last.ckpt`. The file is self-contained: `state_dict` holds that frozen base plus the adapters, `ema_state_dict` holds the EMA'd adapters |
| `renderer.ckpt` | your `weights/` first, else `cbsjtu01/IMTalker` — the same 2121.1 MB file, so the fallback is not a downgrade |
| `wav2vec2-base-960h/` | your `weights/` first, else `cbsjtu01/IMTalker` |
| `blink_motion.pt` | your `weights/`, else `niloy629/personaplex-parallel-depformer`. Optional — production runs with blinking off |

If a local `generator_last.ckpt` **is** found it is preferred as the base, with the
static-head adapters applied on top of it — which is the pairing they were trained for.

`adapter_last.pt` is **not** needed: it maps PersonaPlex Helium hidden states into
Wav2Vec2's final hidden space, and driving from an audio file means running Wav2Vec2
directly instead.

### Before you start

| | |
|---|---|
| **GPU** | any NVIDIA GPU with compute capability ≥ 7.0 and ~8 GB of VRAM. No int4 kernels, no TensorRT, no PersonaPlex, so Turing works |
| **Disk** | ~12 GB on `/workspace` for the Python env, plus whatever your weights already take |
| **Pod** | the same pod you run `runpod_speech2face.ipynb` on is ideal — the weights are already there |
| **HF token** | no — every repo section 7 uses is public and ungated |

### What the cells do

1. configuration — paths and the sampler knobs; the only cell you normally edit
2. check the GPU, driver and disk
3. system libraries (`ffmpeg`, `libsndfile1`, `libgl1`)
4. clone the repo into `/workspace/speech2face_imtalker`
5. write the inference-only requirement files + `setup_env_imtalker.sh`
6. build the Python 3.10 env in `/workspace/imtalker_only/env` (~5–12 min)
7. resolve the checkpoints (~2.8 GB of public downloads on a fresh pod)
8. write `imtalker_pipeline_infer.py`
9. check the inputs and preview the reference frame
10. generate the video
11. play it inline, and read the head-motion report
12. the freeze knob, the other knobs, and troubleshooting


## 1 · Configuration

`WEIGHTS_DIR` is the one setting worth checking. Leave it `""` and section 7 searches the
usual places (`/workspace/speech2face/weights` first, which is where `prepare_x86.sh` puts
things); set it if your `generator_last.ckpt` lives somewhere else.

The sampler block is copied from `run_x86.sh`. **`FM_CHUNK_FRAMES = 2` is what freezes the
head** — leave it alone unless you are deliberately comparing against the moving-head
behaviour, which section 12 describes.


In [ ]:
import os, subprocess, sys, time, json, shutil, textwrap, glob
from pathlib import Path

# ----------------------------------------------------------------- settings
REPO_URL = "https://github.com/MoshiHead/speech2face_orin_n_original_IMTalker_inference_only.git"
REPO_DIR = "/workspace/speech2face_imtalker"   # the clone lands here
BRANCH   = ""                                  # "" = the repo's default branch
WORK     = "/workspace/imtalker_only"          # env and outputs live here

# Where your avatar checkpoints already are, if any. "" = search the usual places.
WEIGHTS_DIR = ""

# The static-head generator. True = apply the lora_exp4_lookahead_120ms_static_head
# adapters, which is the model trained to hold the head still. Set False to run the base
# generator alone, exactly as run_x86.sh does.
STATIC_HEAD_LORA = True

# inputs -- relative paths are resolved against the clone
IMAGE = "src/imtalker/image/rb.jpeg"
AUDIO = "src/imtalker/audio/rb_wav.wav"
OUT   = "outputs/rb_pipeline.mp4"              # relative paths are resolved against WORK

# ------------------------------------------- production sampler (run_x86.sh values)
FM_CHUNK_FRAMES = 2      # frames emitted per generator call. 2 = production = frozen head.
WAV2VEC_SEC     = 2.0    # generator window in seconds -> 50 frames at 25 fps
NUM_PREV_FRAMES = 10     # frames of previous motion each call is conditioned on
A_CFG_SCALE     = 1.24
NFE             = 3
SEED            = 42
SHARED_NOISE    = True
NOISE_SEED      = 42
MOTION_REF_BLEND          = 0.0   # >0 pulls motion toward the reference latent
NOISE_TEMPORAL_CORR       = 0.0
MOTION_PRIOR_NOISE_BLEND  = 0.0
FPS             = 25.0   # the generator emits 25 fps by construction -- do not change
RENDER_SUB_BATCH   = 16
RENDERER_PRECISION = "fp32"
CRF             = 18

# eye blink: production runs with this OFF (run_x86.sh passes the path but not the flag)
ENABLE_EYE_BLINK = False

# Explicit overrides -- "" = let section 7 resolve it
GENERATOR_OVERRIDE = ""   # a base generator checkpoint of your own
LORA_OVERRIDE      = ""   # a LoRA checkpoint of your own

# public sources
HF_IMTALKER_REPO = "cbsjtu01/IMTalker"                        # renderer.ckpt, wav2vec2-base-960h
HF_AVATAR_REPO   = "niloy629/personaplex-parallel-depformer"  # avatar_assets/blink_motion.pt
HF_LORA_REPO     = "niloy629/imtalker_lora"                   # the static-head generator
LORA_FILE        = "lora_exp4_lookahead_120ms_static_head_last.ckpt"

# ----------------------------------------------------------------- derived paths
WORK = os.path.abspath(WORK)
CKPT_DIR = os.path.join(WORK, "checkpoints")   # only fallback downloads land here
PY_ENV   = os.path.join(WORK, "env", "bin", "python")
IMTALKER_DIR = os.path.join(REPO_DIR, "src", "imtalker")
os.makedirs(WORK, exist_ok=True)
os.makedirs(os.path.join(WORK, "outputs"), exist_ok=True)

os.environ.setdefault("HF_HOME", os.path.join(WORK, "hf"))
# huggingface_hub refuses to download at all when RunPod's images export
# HF_HUB_ENABLE_HF_TRANSFER=1 and the hf_transfer package is not installed
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

def resolve(p, base):
    """Absolute paths pass through; relative ones are resolved against `base`."""
    return p if os.path.isabs(p) else os.path.join(base, p)

# ----------------------------------------------------------------- helper
def sh(cmd, cwd=None, check=True, env=None, quiet=False):
    """Run a shell command, streaming its output into the notebook as it happens."""
    cwd = cwd or (WORK if os.path.isdir(WORK) else os.getcwd())
    if not quiet:
        print("$ " + cmd, flush=True)
    p = subprocess.Popen(["bash", "-c", cmd], cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1,
                         errors="replace", env={**os.environ, **(env or {})})
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError("command failed (exit %d): %s" % (rc, cmd))
    return rc

# every later %%writefile cell is relative to WORK
os.chdir(WORK)

print("repo       :", REPO_URL)
print("clone into :", REPO_DIR)
print("work dir   :", WORK)
print("weights    :", WEIGHTS_DIR or "(search: /workspace/speech2face/weights, <clone>/weights, ...)")
print("env python :", PY_ENV)
print()
print("sample loop: window=%gs (%df)  emit=%df/call  prev=%df"
      % (WAV2VEC_SEC, int(WAV2VEC_SEC * FPS), FM_CHUNK_FRAMES, NUM_PREV_FRAMES))
print("sampler    : a_cfg_scale=%.2f nfe=%d seed=%d shared_noise=%s(seed=%d)"
      % (A_CFG_SCALE, NFE, SEED, SHARED_NOISE, NOISE_SEED))
print("head       : %s%s"
      % ("FROZEN (production loop)" if FM_CHUNK_FRAMES <= 4
         else "will move -- FM_CHUNK_FRAMES is above the production value of 2",
         " + static-head LoRA" if STATIC_HEAD_LORA else " (no LoRA)"))


## 2 · Check the GPU, driver and disk

A driver older than 525 cannot run the CUDA 12.8 wheels. There is no compute-capability
floor of 8.0 here: this is plain fp32 attention and convolutions, with none of the int4
trunk or TensorRT machinery the speech-to-speech server needs.


In [ ]:
sh("nvidia-smi --query-gpu=index,name,compute_cap,memory.total,driver_version "
   "--format=csv || echo 'NO GPU VISIBLE'", cwd="/")
sh("echo; df -h /workspace / 2>/dev/null | sed -n '1,5p'; echo; uname -m", cwd="/")

out = subprocess.run(["bash", "-c", "nvidia-smi --query-gpu=compute_cap,memory.total "
                      "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
if out.strip():
    cap, mem = [x.strip() for x in out.strip().splitlines()[0].split(",")]
    gib = float(mem) / 1024
    print("\ncompute capability %s, %.0f GiB of VRAM" % (cap, gib))
    if float(cap) < 7.0:
        print("!! this pipeline needs compute capability >= 7.0")
    elif gib < 8:
        print("!! under 8 GiB of VRAM -- expect CUDA OOM at 512x512; see section 12")
    else:
        print("ok")

free_gb = shutil.disk_usage("/workspace" if os.path.isdir("/workspace") else "/").free / 2**30
print("%.0f GiB free on the work volume%s" % (free_gb, "" if free_gb >= 12 else "  !! want >= 12 GiB"))


## 3 · System libraries

`ffmpeg` to mux the audio onto the rendered frames, `libsndfile1` for soundfile/librosa,
`libgl1` + `libglib2.0-0` for OpenCV. Already present in most RunPod images, in which case
this is a no-op.


In [ ]:
sh("export DEBIAN_FRONTEND=noninteractive && "
   "apt-get update -qq && "
   "apt-get install -y -qq --no-install-recommends "
   "  git curl ca-certificates bzip2 "
   "  ffmpeg libsndfile1 libgl1 libglib2.0-0 "
   "&& echo '--- installed ---' && ffmpeg -version | head -1", cwd="/")


## 4 · Clone the repo

Three packages under `src/imtalker` are used: `experiments/original_pod_8998/`
(the production flow-matching generator), `generator/` (the Wav2Vec2 subclass, the options
and the LoRA helper) and `renderer/`. Nothing from `src/moshi_pkg` or `src/niloys` is
imported, so no PersonaPlex, Mimi or sphn.

Re-running this cell on an existing clone just fetches and fast-forwards it; nothing in
`/workspace/imtalker_only` and nothing in your `weights/` is touched.


In [ ]:
os.makedirs(os.path.dirname(REPO_DIR), exist_ok=True)
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(REPO_DIR, "already exists - updating")
    # FETCH_HEAD, not origin/HEAD: a --depth 1 clone has no remote HEAD ref
    sh("git -C %s fetch --depth 1 origin %s && git -C %s reset --hard FETCH_HEAD"
       % (REPO_DIR, BRANCH or "HEAD", REPO_DIR), cwd="/", check=False)
else:
    branch = ("--branch " + BRANCH) if BRANCH else ""
    sh("git clone --depth 1 %s %s %s" % (branch, REPO_URL, REPO_DIR), cwd="/")

for need in ("experiments/original_pod_8998/FM.py", "experiments/original_pod_8998/FMT.py",
             "generator/wav2vec2.py", "generator/train_lora.py",
             "generator/options/base_options.py", "renderer/models.py"):
    p = os.path.join(IMTALKER_DIR, need)
    assert os.path.isfile(p), "the clone is missing %s" % p
print("\nproduction generator:", os.path.join(IMTALKER_DIR, "experiments/original_pod_8998/FM.py"))


## 5 · Write the inference-only build files

The repo's `setup_env_x86.sh` builds the environment for the whole speech-to-speech
server: moshi, sphn, TensorRT, fastapi, uvicorn — ~8 GB of wheels, almost none of which
this notebook imports. The three files below are its inference-only counterpart, written
into `/workspace/imtalker_only` so the repo's own scripts stay untouched.

| file | what it is for |
|---|---|
| `requirements-imtalker-torch.txt` | torch 2.8.0 / torchvision 0.23.0 / torchaudio 2.8.0, CUDA 12.8 builds — the same versions the Jetson and x86 builds pin |
| `requirements-imtalker.txt` | transformers 4.30.2, timm, torchdiffeq, librosa, OpenCV, numpy 1.26.4 — and nothing else |
| `setup_env_imtalker.sh` | Miniforge x86_64 + a Python 3.10 env in `./env`, then the two files above |

**Why Python 3.10 and transformers 4.30.2:** `src/imtalker/generator/wav2vec2.py`
subclasses `Wav2Vec2Model` and calls into `.feature_extractor`, `.feature_projection`,
`._mask_hidden_states` and `.encoder` directly — and this notebook uses two of its own
methods, `extract_projected_frontend` and `encode_from_projected_frontend`. Newer
transformers majors rename parts of that internal surface, and 4.30.2 / tokenizers 0.13.3
publish no wheels for Python 3.12+.


In [ ]:
%%writefile requirements-imtalker-torch.txt
# torch stack for IMTalker inference on any x86_64 NVIDIA GPU -- CUDA 12.8 builds.
# cu128 covers Ampere (A100/A6000), Ada (4090/L40S) and Blackwell (5090/B200, sm_120).
# Same versions as the repo's Jetson and x86 builds, so the checkpoints behave identically.
#     pip install -r requirements-imtalker-torch.txt
--index-url https://download.pytorch.org/whl/cu128
--extra-index-url https://pypi.org/simple
torch==2.8.0+cu128
torchaudio==2.8.0+cu128
torchvision==0.23.0+cu128


In [ ]:
%%writefile requirements-imtalker.txt
# Everything IMTalker inference needs, and nothing the speech-to-speech server needs.
# Install AFTER requirements-imtalker-torch.txt (setup_env_imtalker.sh does both).
#
# The versions are the ones the repo's own requirements.txt pins, so generator.ckpt and
# renderer.ckpt load into exactly the modules they were saved from. The two that matter:
#   transformers 4.30.2 + tokenizers 0.13.3 -- src/imtalker/generator/wav2vec2.py
#       subclasses Wav2Vec2Model and reaches into .feature_extractor, .feature_projection,
#       ._mask_hidden_states and .encoder directly; newer majors rename parts of that.
#   numpy 1.26.4 -- opencv 4.9 and transformers 4.30 are built against the numpy 1.x ABI.
numpy==1.26.4
transformers==4.30.2
tokenizers==0.13.3
huggingface_hub==0.24.7
safetensors==0.4.5
timm==1.0.9
einops==0.7.0
torchdiffeq==0.2.5
librosa==0.11.0
soundfile==0.14.0
numba==0.61.2
llvmlite==0.44.0
opencv-python-headless==4.9.0.80
Pillow==10.4.0
scipy==1.15.3
scikit-image==0.25.2
matplotlib==3.8.4
tqdm==4.67.1


In [ ]:
%%writefile setup_env_imtalker.sh
#!/usr/bin/env bash
# setup_env_imtalker.sh -- Python 3.10 environment for IMTalker inference only.
#
#   ./setup_env_imtalker.sh
#
# This is a trimmed cousin of the repo's setup_env_x86.sh. It installs Miniforge
# (conda-forge, free) into ./.miniforge, creates a Python 3.10 env in ./env, then
# torch/torchaudio/torchvision for CUDA 12.8 followed by requirements-imtalker.txt.
# Nothing from the speech-to-speech server is installed: no moshi, no sphn, no
# TensorRT, no fastapi/uvicorn. Downloads ~4 GB, takes ~5-12 min.
#
# Python 3.10 specifically: transformers 4.30.2 / tokenizers 0.13.3 -- which is what
# IMTalker's Wav2Vec2 subclass expects -- have no wheels for 3.12+.
#
# Everything lands under this directory, so on RunPod it belongs on /workspace and
# survives a pod restart. Safe to re-run: finished steps are skipped.
set -euo pipefail
HERE="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"; cd "$HERE"
MF="$HERE/.miniforge"; ENV="$HERE/env"
MF_URL="${MF_URL:-https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh}"
# keep every cache next to the env so it lands on the persistent volume, not the container disk
export CONDA_PKGS_DIRS="$HERE/.cache/conda-pkgs" PIP_CACHE_DIR="$HERE/.cache/pip" TMPDIR="$HERE/.cache/tmp"
mkdir -p "$CONDA_PKGS_DIRS" "$PIP_CACHE_DIR" "$TMPDIR"
log(){ printf '\n\033[1m==> %s\033[0m\n' "$*"; }
warn(){ printf '\033[33mWARNING: %s\033[0m\n' "$*" >&2; }
die(){ printf '\033[31mERROR: %s\033[0m\n' "$*" >&2; exit 1; }

log "1/7  checks"
[ "$(uname -m)" = x86_64 ] || die "this script is for x86_64 (on a Jetson use the repo's setup_env.sh)"
command -v nvidia-smi >/dev/null || die "nvidia-smi not found -- no NVIDIA driver visible in this container"
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader | sed 's/^/    /'

log "2/7  Miniforge -> $MF"
if [ ! -x "$MF/bin/conda" ]; then
  curl -fL --retry 3 -o "$TMPDIR/miniforge.sh" "$MF_URL" || die "could not download Miniforge from $MF_URL"
  bash "$TMPDIR/miniforge.sh" -b -p "$MF" > "$TMPDIR/miniforge_install.log" 2>&1 \
    || { tail -20 "$TMPDIR/miniforge_install.log"; die "Miniforge install failed"; }
  rm -f "$TMPDIR/miniforge.sh"
fi
"$MF/bin/conda" --version | sed 's/^/    /'

log "3/7  Python 3.10 env -> $ENV"
if [ ! -x "$ENV/bin/python" ]; then
  "$MF/bin/conda" create -y -q -p "$ENV" python=3.10 pip > "$TMPDIR/conda_create.log" 2>&1 \
    || { tail -20 "$TMPDIR/conda_create.log"; die "conda create failed"; }
fi
PY="$ENV/bin/python"
"$PY" -c 'import sys; assert sys.version_info[:2] == (3, 10), sys.version' || die "env python is not 3.10"
"$PY" -m pip install -q --upgrade pip

log "4/7  torch / torchvision / torchaudio for CUDA 12.8 (~3 GB)"
if ! "$PY" -c 'import torch, sys; sys.exit(0 if torch.version.cuda and torch.cuda.is_available() else 1)' 2>/dev/null; then
  "$PY" -m pip install -r "$HERE/requirements-imtalker-torch.txt" \
    || die "torch install failed (index: https://download.pytorch.org/whl/cu128)"
fi
"$PY" -c 'import torch, sys; print("    torch", torch.__version__, "CUDA", torch.version.cuda, "|", torch.cuda.get_device_name(0)); sys.exit(0 if torch.cuda.is_available() else 1)' \
  || die "torch cannot see the GPU"

log "5/7  IMTalker inference requirements"
"$PY" -m pip install -r "$HERE/requirements-imtalker.txt" || die "pip install -r requirements-imtalker.txt failed"

log "6/7  face-alignment (optional: only the face detector behind --crop)"
# --no-deps is deliberate. face-alignment declares plain `numpy` and `opencv-python` with
# no upper bound, so a normal install upgrades numpy to 2.x -- which the opencv 4.9 wheel
# cannot load ("numpy 2.x cannot be run in ... _ARRAY_API not found") -- and drops the GUI
# opencv on top of the headless one. Everything face-alignment actually imports (torch,
# numpy, scipy, scikit-image, tqdm, numba, cv2) is already installed by the step above.
# A failure here is not fatal: imtalker_infer.py falls back to a centre crop, and a
# 512x512 portrait that is already framed on the face needs no crop at all.
"$PY" -m pip install --no-deps "face-alignment==1.4.1" >/dev/null 2>&1 \
  && echo "    face-alignment 1.4.1 (optional --crop support): ok" \
  || warn "face-alignment did not install -- run without --crop, or pre-crop the image yourself"

log "7/7  pin repair (undo anything an earlier run pulled out from under the pins)"
# The GUI opencv overwrites the headless build's cv2, so if it is present replace it.
if "$PY" -m pip show opencv-python >/dev/null 2>&1 || "$PY" -m pip show opencv-contrib-python >/dev/null 2>&1; then
  echo "    removing the GUI opencv and restoring opencv-python-headless"
  "$PY" -m pip uninstall -y -q opencv-python opencv-contrib-python >/dev/null 2>&1 || true
  "$PY" -m pip install -q --force-reinstall --no-deps "opencv-python-headless==4.9.0.80" \
    || die "could not reinstall opencv-python-headless"
fi
# numpy must stay on the 1.x ABI: opencv 4.9 and transformers 4.30 are compiled against it.
if ! "$PY" -c 'import numpy, sys; sys.exit(0 if numpy.__version__ == "1.26.4" else 1)' 2>/dev/null; then
  echo "    restoring numpy==1.26.4"
  "$PY" -m pip install -q --force-reinstall --no-deps "numpy==1.26.4" || die "could not pin numpy to 1.26.4"
fi
echo "    pins ok"

"$PY" - <<'T' || die "installed, but the environment check failed -- re-run this cell once; it repairs the pins"
import sys
import torch, torchvision, transformers, timm, torchdiffeq, librosa, cv2, numpy, PIL
assert torch.cuda.is_available(), "torch has no CUDA"
assert numpy.__version__.startswith("1."), (
    f"numpy {numpy.__version__} is on the 2.x ABI, which the opencv 4.9 and transformers "
    "4.30 wheels cannot load. Re-run this cell: step 7/7 pins it back to 1.26.4."
)
assert transformers.__version__ == "4.30.2", f"transformers {transformers.__version__} (want 4.30.2)"
try:
    import face_alignment
    fa = f" | face_alignment {face_alignment.__version__}"
except Exception:
    fa = " | face_alignment MISSING (--crop unavailable)"
cc = torch.cuda.get_device_capability(0)
print(f"    python {sys.version.split()[0]} | torch {torch.__version__} (CUDA {torch.version.cuda}) | "
      f"transformers {transformers.__version__} | numpy {numpy.__version__} | "
      f"opencv {cv2.__version__}{fa}")
print(f"    {torch.cuda.get_device_name(0)} sm_{cc[0]}{cc[1]} "
      f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB")
T
rm -rf "$TMPDIR"
log "ENV READY: $ENV"


## 6 · Build the Python 3.10 environment

~5–12 minutes and ~4 GB the first time. Safe to re-run: Miniforge, the env and every
finished pip step are skipped, so this is also the cell to re-run after a pod restart to
confirm the env on the volume still works.

Two things the script does that are worth knowing about, because getting them wrong breaks
the env in a way that only shows up much later:

* **`face-alignment` is installed with `--no-deps`.** It declares plain `numpy` and
  `opencv-python` with no upper bound, so a normal install upgrades numpy to 2.x — which
  the opencv 4.9 and transformers 4.30 wheels cannot load (`_ARRAY_API not found`) — and
  drops the GUI OpenCV on top of the headless one. Everything it imports is already
  installed by the step before it. It is also still allowed to fail: this pipeline does no
  face detection at all (the live server feeds `weights/ref.jpeg` straight in), so
  face-alignment is only there for the optional preview crop in section 9.
* **step 7/7 is a pin repair.** It puts numpy back on 1.26.4 and restores
  `opencv-python-headless` if anything moved them, so re-running this cell fixes an env a
  stray `pip install` has broken rather than making you rebuild it.


In [ ]:
sh("chmod +x setup_env_imtalker.sh && ./setup_env_imtalker.sh")
assert os.path.isfile(PY_ENV), "the env was not created at %s" % PY_ENV
print("\nenv python:", PY_ENV)


## 7 · Resolve the checkpoints

On a fresh pod `prepare_x86.sh` downloads nothing avatar-related, so this cell resolves
everything from public repos, preferring anything it finds locally.

**The generator.** `niloy629/imtalker_lora` holds
`lora_exp4_lookahead_120ms_static_head_last.ckpt` (558 MB), and its Lightning
`hyper_parameters` identify it exactly:

```
exp_name   = 'lora_exp4_lookahead_120ms_static_head'
base_ckpt  = '/workspace/exps/exp4_lookahead_120ms/checkpoints/last.ckpt'
lora_rank  = 16      lora_alpha = 32      lora_dropout = 0.05
no_lora_pose_projection = False    no_lora_audio_projection = False
only_lora_pose_projection = False
wav2vec_sec = 2      dim_w = 32    fmt_depth = 8    num_prev_frames = 10
```

`base_ckpt` is the run whose `last.ckpt` is your `generator_last.ckpt`, and the adapters
were trained on top of it with the base frozen. That last detail is what makes this
usable on a pod that has none of your files: the checkpoint's `state_dict` carries those
320 frozen base tensors **and** the 92 adapters, while `ema_state_dict` carries only the
EMA'd adapters. One file, a complete static-head generator.

The 92 adapters cover 46 modules — `attn.qkv`, `attn.proj`, `mlp.fc1`, `mlp.fc2` and
`adaLN_modulation.1` across all 8 FMT blocks, the two `fmt.decoder` linears, and the
`audio` / `pose` / `gaze` / `cam` projections — which is exactly what
`generator.train_lora.lora_target_names` selects with the default include flags, so they
load without a single mismatch. Section 8 reads the rank out of the checkpoint rather
than trusting a default, because `load_state_dict` raises on a shape mismatch even with
`strict=False`.

`renderer.ckpt` and `wav2vec2-base-960h/` prefer your `weights/` and otherwise come from
`cbsjtu01/IMTalker`. That renderer is 2121.1 MB, the same size as the copy in
`niloy629/bangla-gemma-s2s/imtalker_renderer.ckpt`, so the fallback is the file your
pipeline already uses rather than a substitute.


In [ ]:
os.makedirs(CKPT_DIR, exist_ok=True)

SEARCH_DIRS = [d for d in ([WEIGHTS_DIR] if WEIGHTS_DIR else [
    "/workspace/speech2face/weights",          # where a prepared pod keeps them
    os.path.join(REPO_DIR, "weights"),
    "/workspace/weights",
    "/workspace/IMTalker/checkpoints",
    CKPT_DIR,
]) if d]

def find_weight(name, extra=()):
    """First hit wins; a directory counts when it is non-empty."""
    for d in list(SEARCH_DIRS) + list(extra):
        p = os.path.join(d, name)
        if os.path.isdir(p) and os.listdir(p):
            return p
        if os.path.isfile(p) and os.path.getsize(p) > 1024:
            return p
    return ""

def hf_fetch(repo, files, dest):
    dl = textwrap.dedent("""
        import os, sys
        from huggingface_hub import hf_hub_download
        repo, dest, files = sys.argv[1], sys.argv[2], sys.argv[3:]
        for name in files:
            out = os.path.join(dest, name)
            if os.path.exists(out) and os.path.getsize(out) > 1024:
                print("    have   %s (%.1f MB)" % (name, os.path.getsize(out) / 1e6))
                continue
            print("    get    %s ..." % name, flush=True)
            hf_hub_download(repo_id=repo, filename=name, repo_type="model",
                            local_dir=dest, local_dir_use_symlinks=False)
            print("    stored %s (%.1f MB)" % (name, os.path.getsize(out) / 1e6), flush=True)
    """)
    Path(os.path.join(WORK, "_fetch_ckpts.py")).write_text(dl, encoding="utf-8")
    sh('"%s" _fetch_ckpts.py %s %s %s' % (PY_ENV, repo, dest, " ".join(files)))

print("searching:", *SEARCH_DIRS, sep="\n  ")
print()

# ---------------------------------------------------------------- generator + LoRA
GEN_LOCAL  = find_weight("generator_last.ckpt") or find_weight("generator.ckpt")
LORA_LOCAL = find_weight(LORA_FILE)

def fetch_static_head_lora():
    global LORA_LOCAL
    if not LORA_LOCAL:
        print("static-head generator -> %s" % HF_LORA_REPO)
        hf_fetch(HF_LORA_REPO, [LORA_FILE], CKPT_DIR)
        LORA_LOCAL = os.path.join(CKPT_DIR, LORA_FILE)
    return LORA_LOCAL

if GENERATOR_OVERRIDE:
    GENERATOR_PATH, LORA_PATH = GENERATOR_OVERRIDE, LORA_OVERRIDE
    source = "GENERATOR_OVERRIDE"
elif GEN_LOCAL:
    GENERATOR_PATH = GEN_LOCAL
    LORA_PATH = (LORA_OVERRIDE or fetch_static_head_lora()) if STATIC_HEAD_LORA else ""
    source = "your base" + (" + static-head LoRA" if LORA_PATH else " (no LoRA, = run_x86.sh)")
elif STATIC_HEAD_LORA:
    # self-contained: state_dict holds the frozen base, ema_state_dict the adapters
    GENERATOR_PATH, LORA_PATH = fetch_static_head_lora(), ""
    source = "static-head checkpoint (base + adapters in one file)"
else:
    print("STATIC_HEAD_LORA is False and no local base generator was found -> "
          "falling back to the upstream base from %s" % HF_IMTALKER_REPO)
    hf_fetch(HF_IMTALKER_REPO, ["generator.ckpt"], CKPT_DIR)
    GENERATOR_PATH, LORA_PATH = os.path.join(CKPT_DIR, "generator.ckpt"), ""
    source = "upstream base, NO static-head LoRA -- the head will move"

# ---------------------------------------------------------------- renderer + wav2vec2
RENDERER_PATH = find_weight("renderer.ckpt")
WAV2VEC_PATH  = find_weight("wav2vec2-base-960h")
if not RENDERER_PATH or not WAV2VEC_PATH:
    want = []
    if not RENDERER_PATH:
        want.append("renderer.ckpt")
    if not WAV2VEC_PATH:
        want += ["wav2vec2-base-960h/config.json",
                 "wav2vec2-base-960h/preprocessor_config.json",
                 "wav2vec2-base-960h/feature_extractor_config.json",
                 "wav2vec2-base-960h/pytorch_model.bin"]
    print("\nrenderer/wav2vec2 not found locally -> %s" % HF_IMTALKER_REPO)
    hf_fetch(HF_IMTALKER_REPO, want, CKPT_DIR)
    RENDERER_PATH = RENDERER_PATH or find_weight("renderer.ckpt", extra=[CKPT_DIR])
    WAV2VEC_PATH  = WAV2VEC_PATH  or find_weight("wav2vec2-base-960h", extra=[CKPT_DIR])

# ---------------------------------------------------------------- blink (optional)
BLINK_PATH = find_weight("blink_motion.pt")
if ENABLE_EYE_BLINK and not BLINK_PATH:
    print("\nblink_motion.pt -> %s" % HF_AVATAR_REPO)
    hf_fetch(HF_AVATAR_REPO, ["avatar_assets/blink_motion.pt"], CKPT_DIR)
    BLINK_PATH = os.path.join(CKPT_DIR, "avatar_assets", "blink_motion.pt")

assert os.path.isfile(GENERATOR_PATH), "generator checkpoint missing: %s" % GENERATOR_PATH
assert os.path.isfile(RENDERER_PATH), "renderer.ckpt missing: %s" % RENDERER_PATH
assert os.path.isfile(os.path.join(WAV2VEC_PATH, "config.json")), "wav2vec2-base-960h incomplete"

if LORA_PATH:
    lora_note = LORA_PATH
elif STATIC_HEAD_LORA:
    lora_note = "(adapters are inside the generator file)"
else:
    lora_note = "(none)"

print("\nresolved (%s):" % source)
for label, p in (("generator", GENERATOR_PATH), ("lora", lora_note),
                 ("renderer", RENDERER_PATH), ("wav2vec2", WAV2VEC_PATH),
                 ("blink", BLINK_PATH if ENABLE_EYE_BLINK else "(disabled)")):
    if p and os.path.isfile(p):
        print("  %-10s %s  (%.0f MB)" % (label, p, os.path.getsize(p) / 1e6))
    else:
        print("  %-10s %s" % (label, p or "(none)"))
if not STATIC_HEAD_LORA:
    print("\n!! STATIC_HEAD_LORA is False: the static-head adapters are NOT applied.")


## 8 · The inference script

`imtalker_pipeline_infer.py` is the avatar half of `imtalker_native_passthrough.py` — the
module `run_x86.sh` actually serves — lifted out of the websocket server and driven from
an audio file. The parts that decide how the face moves are kept identical:

* **the production generator.** `experiments.original_pod_8998.FM`, not `generator.FM`.
  They are not interchangeable: only the production one accepts `noise_init`,
  `noise_temporal_corr`, `motion_prior_noise_blend` and the RoPE `global_offset`, and only
  it is what `generator_last.ckpt` was trained against.
* **the production weight loaders**, with one necessary refinement. Production picks
  `ema_state_dict` first; that is right for a full fine-tune, but a LoRA run's
  `ema_state_dict` holds *only* the 92 EMA'd adapter tensors and no model at all. The
  script therefore takes `ema_state_dict` as the base only when it contains non-adapter
  keys, otherwise `state_dict` — then injects the LoRA wrappers with
  `generator.train_lora.apply_lora_to_model` and overlays the EMA'd adapters on top,
  which is what the live server does through `--lora_generator_path`. The rank and alpha
  come from the checkpoint's own `hyper_parameters` (or off a `lora_A.weight` shape),
  because a wrong rank makes `load_state_dict` raise even with `strict=False`.
* **the reference image path**: 512×512 LANCZOS → `ToTensor`, encoded once, no face
  detection and no cropping — what the live server does with `weights/ref.jpeg`.
* **the streaming sample loop, which is what freezes the head.** The generator runs on a
  50-frame window (`wav2vec_sec 2.0 × 25 fps`) but only `fm_chunk_frames` = 2 frames are
  kept per call. `stream_state` carries 10 frames of previously generated motion and the
  previous initial noise into the next call, and `noise_init` comes from one shared noise
  buffer indexed by absolute frame. Every call is therefore anchored to where the face
  already was, which pins the low-frequency component — head pose and tilt — while the
  mouth still follows the 2 fresh audio frames.
* **the renderer call and output range**: `adapt → latent_token_decoder → decode`, then
  `clamp(0,1)`, exactly as `_render_motion` does.

What necessarily differs is the audio front end. Production feeds PersonaPlex Helium
hidden states through `adapter_last.pt` into Wav2Vec2's final hidden space. An audio file
has no Helium states, so this script runs Wav2Vec2 itself —
`extract_projected_frontend` → `encode_from_projected_frontend`, the same two calls the
live engine uses for its own session dumps — resamples 50 Hz → 25 Hz with the same
`align_corners=False` interpolation, and passes the result in as `a_feat`. That is the
space the adapter was trained to predict, so the generator sees the distribution it
expects.

Checkpoints are opened with `weights_only=True` first and the full unpickler as a
fallback, since torch 2.8 flipped that default and a Lightning checkpoint can trip over it.

Two practical additions over the live code: frames are streamed to disk as they are
rendered rather than stacked in VRAM, and the script prints a **head-motion report** —
per-dimension standard deviation of the motion latent and the frame-to-frame delta — so
you can confirm numerically that the head is still rather than squinting at the video.


In [ ]:
%%writefile imtalker_pipeline_infer.py
#!/usr/bin/env python3
"""Offline inference with THIS repo's production avatar pipeline -- frozen head.

This is the avatar half of `src/niloys/live/imtalker_native_passthrough.py` (what
`run_x86.sh` actually serves), lifted out of the websocket server and driven from an
audio file instead of from PersonaPlex. Everything that shapes the motion is kept
identical to production:

  * `experiments.original_pod_8998.FM.FMGenerator` -- the production generator, not
    `generator.FM`. The two differ: only this one accepts `noise_init`,
    `noise_temporal_corr`, `motion_prior_noise_blend` and the RoPE `global_offset`.
  * the production checkpoint loaders (`ema_state_dict` first, then `state_dict`,
    then `model`, with the `model.` / `gen.` prefixes stripped).
  * `renderer.models.IMTRenderer` from `renderer.ckpt`, fp32, output clamped to [0,1].
  * the reference image at 512x512 LANCZOS -> ToTensor, encoded once.
  * **the streaming sample loop, which is what freezes the head.** Production runs
    `wav2vec_sec 2.0` (a 50-frame generator window) but emits only `fm_chunk_frames`
    frames per call -- 2 frames, 80 ms -- and carries `stream_state` plus a shared,
    pre-generated noise buffer indexed by absolute frame. Each call is therefore
    anchored by 10 frames of previously generated motion and 25 frames of the previous
    initial noise, so the low-frequency component (head pose, tilt) cannot drift while
    the mouth still tracks the fresh audio. Sampling the whole clip in one 50-frame
    shot -- what upstream IMTalker's `generate.py` does -- is exactly what brings the
    head movement back.

The one part that cannot be identical is the audio front end. Production feeds
PersonaPlex Helium hidden states through `adapter_last.pt` into Wav2Vec2's final hidden
space; an audio file has no Helium states, so this script runs Wav2Vec2 itself
(`extract_projected_frontend` -> `encode_from_projected_frontend`, the same two calls
the live engine uses for its own dumps) and feeds the result in as `a_feat`. That is
the space the adapter was trained to predict, so the generator sees what it expects.

    python imtalker_pipeline_infer.py \
        --imtalker_dir /workspace/speech2face_imtalker/src/imtalker \
        --image  .../rb.jpeg \
        --audio  .../rb_wav.wav \
        --out    .../rb_pipeline.mp4 \
        --generator_path .../generator_last.ckpt \
        --renderer_path  .../renderer.ckpt \
        --wav2vec_model_path .../wav2vec2-base-960h \
        --fm_chunk_frames 2 --wav2vec_sec 2.0 --a_cfg_scale 1.24 --nfe 3 \
        --shared_noise --noise_seed 42 --fp32 --tf32
"""
from __future__ import annotations

import argparse
import json
import shutil
import subprocess
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from transformers import Wav2Vec2FeatureExtractor

WAV2VEC_SR = 16_000


# --------------------------------------------------------------------------- paths
def add_imtalker_to_path(imtalker_dir: Path) -> None:
    """One entry, exactly as the live module does it: `experiments.original_pod_8998.FM`
    imports `generator.wav2vec2`, and `generator.options.base_options` resolves from the
    same root, so src/imtalker alone is enough."""
    s = str(imtalker_dir.resolve())
    if s not in sys.path:
        sys.path.insert(0, s)


# --------------------------------------------------------------------------- options
def build_opt(args: argparse.Namespace):
    """Model hyper-parameters come from the repo's own BaseOptions, overridden with the
    values run_x86.sh passes, so the option namespace the models see matches production."""
    from generator.options.base_options import BaseOptions

    argv = [
        "imtalker_pipeline_infer.py",
        "--input_size", "512",
        "--fps", str(args.fps),
        "--sampling_rate", str(WAV2VEC_SR),
        "--wav2vec_sec", str(args.wav2vec_sec),
        "--wav2vec_model_path", str(args.wav2vec_model_path),
        "--num_prev_frames", str(args.num_prev_frames),
        "--nfe", str(args.nfe),
        "--a_cfg_scale", str(args.a_cfg_scale),
        "--seed", str(args.seed),
        "--torchdiffeq_ode_method", args.ode_method,
    ]
    old_argv, sys.argv = sys.argv, argv
    try:
        opt = BaseOptions().parse()
    finally:
        sys.argv = old_argv

    opt.rank = args.device          # FMGenerator._device() reads this
    opt.device = args.device
    opt.ngpus = 1
    opt.generator_path = str(args.generator_path)
    opt.renderer_path = str(args.renderer_path)
    opt.lora_generator_path = str(args.lora_generator_path or "")
    opt.lora_dropout = float(args.lora_dropout)
    opt.no_lora_pose_projection = bool(args.no_lora_pose_projection)
    opt.no_lora_audio_projection = bool(args.no_lora_audio_projection)
    opt.only_lora_pose_projection = bool(args.only_lora_pose_projection)
    # run_x86.sh passes --skip_fm_audio_encoder: the generator's own raw-waveform
    # Wav2Vec2 branch is never built, because features arrive precomputed. Keep that,
    # and run Wav2Vec2 separately below, so FMGenerator is constructed as in production.
    opt.skip_fm_audio_encoder = True
    # production leaves all three of these at 0.0 (run_x86.sh), but they are what the
    # production FM exposes for damping motion further, so keep them wired
    opt.noise_temporal_corr = float(args.noise_temporal_corr)
    opt.motion_prior_noise_blend = float(args.motion_prior_noise_blend)
    opt.motion_ref_blend = float(args.motion_ref_blend)
    opt.disable_rope_global_offset = False
    opt.render_sub_batch = int(args.render_sub_batch)
    opt.enable_eye_blink_composite = bool(args.enable_eye_blink_composite)
    opt.blink_motion_path = str(args.blink_motion_path or "")
    opt.eye_left_x = float(args.eye_left_x)
    opt.eye_right_x = float(args.eye_right_x)
    opt.eye_center_y = float(args.eye_center_y)
    opt.eye_radius_x = float(args.eye_radius_x)
    opt.eye_radius_y = float(args.eye_radius_y)
    opt.eye_feather = float(args.eye_feather)
    return opt


# --------------------------------------------------------------------------- weights
def torch_load(path: str):
    """These are the user's own checkpoints. torch 2.8 defaults to weights_only=True,
    which a Lightning checkpoint can trip over, so fall back to the full unpickler --
    the same call the live server makes."""
    try:
        return torch.load(path, map_location="cpu")
    except Exception:
        return torch.load(path, map_location="cpu", weights_only=False)


def strip_model_prefix(d: dict) -> dict:
    return {k.replace("model.", "", 1) if k.startswith("model.") else k: v for k, v in d.items()}


def clean_generator_state(ckpt) -> dict:
    """Verbatim from imtalker_native_passthrough._clean_generator_state: EMA first.
    Used for the LoRA side, where that is exactly what you want."""
    raw = ckpt.get("ema_state_dict") or ckpt.get("state_dict", ckpt.get("model", ckpt))
    if isinstance(raw, dict) and "model" in raw and isinstance(raw["model"], dict):
        raw = raw["model"]
    return strip_model_prefix(raw)


def base_generator_state(ckpt) -> dict:
    """The *full* model weights, which is not the same selection as above.

    A Lightning LoRA run keeps only the EMA of the trainable LoRA tensors in
    `ema_state_dict` -- 92 tensors, no base weights at all -- while `state_dict` holds
    the complete model: the frozen base, the LoRA adapters and the audio encoder. So for
    the base we prefer `state_dict` and fall back to `ema_state_dict` only if there is
    nothing else, which is the case for a full fine-tune checkpoint.
    """
    if not isinstance(ckpt, dict):
        return ckpt
    candidates: list[tuple[str, dict]] = []
    for key in ("ema_state_dict", "state_dict", "model"):
        raw = ckpt.get(key)
        if isinstance(raw, dict) and raw:
            if isinstance(raw.get("model"), dict):
                raw = raw["model"]
            candidates.append((key, strip_model_prefix(raw)))
    if not candidates:
        return strip_model_prefix(ckpt)
    # Production prefers ema_state_dict, so keep that -- but only when it is a whole
    # model. In a LoRA run it holds nothing but the 92 adapter tensors.
    for key, state in candidates:
        if key == "ema_state_dict" and any("lora_" not in k for k in state):
            return state
    for key, state in candidates:
        if key != "ema_state_dict":
            return state
    return candidates[0][1]


def lora_hyper_params(ckpt, state: dict, args: argparse.Namespace) -> tuple[int, float, float]:
    """rank/alpha/dropout, from the checkpoint itself where possible.

    Getting the rank wrong is not a soft failure: `load_state_dict` raises on a shape
    mismatch even with strict=False. The static-head checkpoint records
    lora_rank=16 / lora_alpha=32 in its Lightning `hyper_parameters`, and the rank is
    also readable off any `lora_A.weight` as its first dimension. An explicit
    --lora_rank / --lora_alpha on the command line still wins.
    """
    rank = alpha = dropout = None
    hp = ckpt.get("hyper_parameters") if isinstance(ckpt, dict) else None
    if isinstance(hp, dict):
        rank, alpha, dropout = hp.get("lora_rank"), hp.get("lora_alpha"), hp.get("lora_dropout")
    if rank is None:
        for key, value in state.items():
            if key.endswith("lora_A.weight") and hasattr(value, "shape"):
                rank = int(value.shape[0])
                break
    if args.lora_rank:
        rank = int(args.lora_rank)
    if args.lora_alpha:
        alpha = float(args.lora_alpha)
    rank = int(rank or 16)
    alpha = float(alpha) if alpha else 2.0 * rank
    dropout = float(dropout if dropout is not None else args.lora_dropout)
    return rank, alpha, dropout


def load_fm(opt, args: argparse.Namespace, device: torch.device):
    from experiments.original_pod_8998.FM import FMGenerator

    t0 = time.perf_counter()
    fm = FMGenerator(opt).to(device).eval()

    gen_ckpt = torch_load(opt.generator_path)
    base = base_generator_state(gen_ckpt)
    lora_path = str(getattr(opt, "lora_generator_path", "") or "")
    lora_ckpt = torch_load(lora_path) if lora_path else None

    def inject_lora() -> None:
        from generator.train_lora import apply_lora_to_model

        rank, alpha, dropout = lora_hyper_params(
            lora_ckpt if lora_ckpt is not None else gen_ckpt, base, args
        )
        apply_lora_to_model(
            fm,
            rank=rank,
            alpha=alpha,
            dropout=dropout,
            include_pose_lora=not bool(opt.no_lora_pose_projection),
            include_audio_lora=not bool(opt.no_lora_audio_projection),
            only_pose_lora=bool(opt.only_lora_pose_projection),
        )

    # Order matters, and it depends on the base checkpoint.
    #
    # A static-head LoRA checkpoint is self-contained: its state_dict is already in the
    # wrapped layout (`fmt.blocks.0.attn.qkv.base.weight` next to `...lora_A.weight`), so
    # the wrappers have to exist before it is loaded.
    #
    # A plain base checkpoint has the unwrapped names (`fmt.blocks.0.attn.qkv.weight`),
    # which would miss every wrapped module. That one is loaded first and wrapped after --
    # `apply_lora_to_model` wraps the live nn.Linear objects and keeps their weights, which
    # is the order the live server's _load_fm uses.
    wrapped = any(".base." in k for k in base)
    if wrapped:
        inject_lora()

    missing, unexpected = fm.load_state_dict(base, strict=False)
    matched = len(base) - len(unexpected)
    print(
        f"[FM] {opt.generator_path}: matched {matched}/{len(base)} tensors "
        f"(missing={len(missing)} unexpected={len(unexpected)}) "
        f"layout={'lora-wrapped' if wrapped else 'plain'}",
        flush=True,
    )
    if missing:
        print(f"[FM] first missing: {missing[:5]}", flush=True)
    if matched <= 0:
        raise RuntimeError(f"no generator weight matched the model: {opt.generator_path}")

    if not wrapped and lora_ckpt is not None:
        inject_lora()

    # Overlay the EMA'd LoRA adapters -- what the live server loads through
    # --lora_generator_path. When base and adapters live in the same checkpoint, that is
    # this file's own ema_state_dict.
    ema_src = lora_ckpt if lora_ckpt is not None else gen_ckpt
    ema = ema_src.get("ema_state_dict") if isinstance(ema_src, dict) else None
    if lora_ckpt is not None and not ema:
        ema = base_generator_state(lora_ckpt)
    if ema:
        ema = strip_model_prefix(ema)
        _, unexpected_e = fm.load_state_dict(ema, strict=False)
        lora_keys = sum(1 for key in ema if "lora_" in key)
        print(
            f"[FM] LoRA overlay from {lora_path or opt.generator_path}: "
            f"applied {len(ema) - len(unexpected_e)}/{len(ema)} tensors "
            f"({lora_keys} lora_*), unexpected={len(unexpected_e)}",
            flush=True,
        )
        if len(ema) - len(unexpected_e) == 0:
            raise RuntimeError(
                "the LoRA adapters matched nothing -- check --lora_rank / the include flags"
            )

    fm.to(device).eval()
    print(f"[FM] ready in {time.perf_counter() - t0:.1f}s", flush=True)
    return fm


def load_renderer(opt, device: torch.device, dtype: torch.dtype):
    from renderer.models import IMTRenderer

    t0 = time.perf_counter()
    renderer = IMTRenderer(opt).to(device).eval()
    ckpt = torch_load(opt.renderer_path)
    raw = ckpt.get("state_dict", ckpt.get("model", ckpt)) if isinstance(ckpt, dict) else ckpt
    cleaned = {k.replace("gen.", "", 1).replace("model.", "", 1): v for k, v in raw.items()}
    missing, unexpected = renderer.load_state_dict(cleaned, strict=False)
    renderer = renderer.to(dtype=dtype).eval()
    matched = len(cleaned) - len(unexpected)
    print(
        f"[renderer] {opt.renderer_path}: matched {matched}/{len(cleaned)} tensors "
        f"(missing={len(missing)} unexpected={len(unexpected)}) dtype={dtype}",
        flush=True,
    )
    if matched <= 0:
        raise RuntimeError(f"no renderer weight matched the model: {opt.renderer_path}")
    print(f"[renderer] ready in {time.perf_counter() - t0:.1f}s", flush=True)
    return renderer


# --------------------------------------------------------------------------- inputs
def load_ref_image(path: Path, device: torch.device, dtype: torch.dtype) -> torch.Tensor:
    """Verbatim from imtalker_native_passthrough.load_ref_image -- no face detection and
    no cropping, because that is what the live server does with weights/ref.jpeg."""
    img = Image.open(path).convert("RGB").resize((512, 512), Image.LANCZOS)
    return T.ToTensor()(img).unsqueeze(0).to(device=device, dtype=dtype)


def load_audio_16k(path: Path) -> tuple[np.ndarray, float]:
    """Mono float32 at 16 kHz. torchaudio first (same resampler as the live engine),
    librosa as the fallback for anything its backends will not open."""
    wav = None
    try:
        import torchaudio

        data, sr = torchaudio.load(str(path))
        if data.ndim == 2 and data.shape[0] > 1:
            data = data.mean(dim=0, keepdim=True)
        if int(sr) != WAV2VEC_SR:
            data = torchaudio.functional.resample(data, int(sr), WAV2VEC_SR)
        wav = data.reshape(-1).contiguous().numpy().astype(np.float32)
    except Exception as exc:  # noqa: BLE001
        print(f"[audio] torchaudio could not read it ({exc}); falling back to librosa", flush=True)
    if wav is None:
        import librosa

        wav = np.asarray(librosa.load(str(path), sr=WAV2VEC_SR, mono=True)[0], dtype=np.float32)
    seconds = float(len(wav)) / WAV2VEC_SR
    print(f"[audio] {path} -> {seconds:.2f}s mono @ {WAV2VEC_SR} Hz ({len(wav)} samples)", flush=True)
    if len(wav) == 0:
        raise ValueError(f"{path} decoded to zero samples")
    return wav, seconds


# --------------------------------------------------------------------------- wav2vec
@torch.no_grad()
def wav2vec_50hz(wav16: np.ndarray, extractor, model, device: torch.device) -> torch.Tensor:
    """768-d Wav2Vec2 final hidden states at ~50 Hz -- the same two calls the live engine
    makes in _extract_wav2vec_raw_50hz, and the space adapter_last.pt predicts into."""
    inputs = extractor(wav16, sampling_rate=WAV2VEC_SR, return_tensors="pt", padding=True)
    kwargs = {"input_values": inputs.input_values.to(device)}
    if getattr(inputs, "attention_mask", None) is not None:
        kwargs["attention_mask"] = inputs.attention_mask.to(device)
    frontend = model.extract_projected_frontend(**kwargs)
    feat = model.encode_from_projected_frontend(frontend).last_hidden_state[0]
    return feat.detach().float().contiguous()


@torch.no_grad()
def wav2vec_50hz_long(
    wav16: np.ndarray, extractor, model, device: torch.device,
    window_sec: float, context_sec: float,
) -> torch.Tensor:
    """Wav2Vec2's encoder attends over the whole input, so a long file is quadratic and
    will OOM. Run it in windows with `context_sec` of real audio on each side and keep
    only the middle, which leaves the kept frames with full receptive field."""
    total = int(len(wav16))
    win = max(1, int(window_sec * WAV2VEC_SR))
    ctx = max(0, int(context_sec * WAV2VEC_SR))
    if total <= win:
        return wav2vec_50hz(wav16, extractor, model, device)

    parts: list[torch.Tensor] = []
    pos = 0
    while pos < total:
        end = min(total, pos + win)
        a, b = max(0, pos - ctx), min(total, end + ctx)
        feat = wav2vec_50hz(wav16[a:b], extractor, model, device)
        n = int(feat.shape[0])
        lo = int(round((pos - a) / float(b - a) * n))
        hi = int(round((end - a) / float(b - a) * n))
        parts.append(feat[lo:max(hi, lo + 1)])
        pos = end
    out = torch.cat(parts, dim=0).contiguous()
    print(f"[wav2vec] {len(parts)} windows of {window_sec:g}s (+{context_sec:g}s context)", flush=True)
    return out


# --------------------------------------------------------------------------- blink
class BlinkCompositor:
    """The optional eye-region map compositing of the live engine. run_x86.sh passes
    --blink_motion_path but NOT --enable_eye_blink_composite, so production has this off;
    it is here because a frozen head with blinking eyes is usually what you want from a
    long clip."""

    def __init__(self, opt, renderer, g_r, device, dtype):
        self.opt = opt
        self.device = device
        self.dtype = dtype
        path = Path(str(opt.blink_motion_path))
        if not path.is_file():
            raise FileNotFoundError(f"blink_motion_path does not exist: {path}")
        motion = self._motion_from_payload(torch_load(str(path)), path, int(opt.dim_w))

        parts: list[list[torch.Tensor]] = []
        chunk = max(1, int(opt.render_sub_batch))
        with torch.no_grad():
            for start in range(0, int(motion.shape[0]), chunk):
                sub = motion[start:start + chunk].to(device, dtype=dtype)
                maps = renderer.latent_token_decoder(renderer.adapt(sub, g_r.expand(int(sub.shape[0]), -1)))
                if not parts:
                    parts = [[] for _ in maps]
                for idx, m in enumerate(maps):
                    parts[idx].append(m.detach())
        self.maps = tuple(torch.cat(p, dim=0).contiguous() for p in parts)
        self.masks = tuple(self._eye_mask(m.shape[-2], m.shape[-1]) for m in self.maps)
        print(
            f"[blink] cached {int(motion.shape[0])} blink frames from {path} "
            f"shapes={[tuple(m.shape) for m in self.maps]}",
            flush=True,
        )

    @staticmethod
    def _motion_from_payload(payload, path: Path, dim_w: int) -> torch.Tensor:
        if isinstance(payload, torch.Tensor):
            motion = payload
        elif isinstance(payload, dict):
            motion = None
            for key in ("motion", "motion_latents", "latents", "full_motion", "pred_motion", "x"):
                value = payload.get(key)
                if isinstance(value, torch.Tensor):
                    motion = value
                    break
            if motion is None:
                candidates = [
                    v for v in payload.values()
                    if isinstance(v, torch.Tensor) and v.ndim >= 2 and v.shape[-1] == dim_w
                ]
                if not candidates:
                    raise ValueError(
                        f"no motion tensor in {path}; keys={list(payload.keys())[:20]}"
                    )
                motion = candidates[0]
        else:
            raise TypeError(f"unsupported blink payload {type(payload)} from {path}")
        motion = motion.detach().float().cpu()
        if motion.ndim == 3 and motion.shape[0] == 1:
            motion = motion[0]
        if motion.ndim != 2 or motion.shape[-1] != dim_w:
            raise ValueError(f"blink motion must be (T,{dim_w}); got {tuple(motion.shape)} from {path}")
        return motion.contiguous()

    def _eye_mask(self, height: int, width: int) -> torch.Tensor:
        yy = torch.linspace(0.0, 1.0, int(height), device=self.device, dtype=self.dtype).view(1, 1, height, 1)
        xx = torch.linspace(0.0, 1.0, int(width), device=self.device, dtype=self.dtype).view(1, 1, 1, width)
        radius_x = max(float(self.opt.eye_radius_x), 1e-6)
        radius_y = max(float(self.opt.eye_radius_y), 1e-6)
        feather = max(float(self.opt.eye_feather), 1e-6)
        mask = torch.zeros(1, 1, height, width, device=self.device, dtype=self.dtype)
        for center_x in (float(self.opt.eye_left_x), float(self.opt.eye_right_x)):
            dist = ((xx - center_x) / radius_x).square() + ((yy - float(self.opt.eye_center_y)) / radius_y).square()
            ellipse = ((1.0 + feather) - dist).clamp(0.0, feather) / feather
            mask = torch.maximum(mask, ellipse)
        return mask.clamp(0.0, 1.0).contiguous()

    def apply(self, current_maps, start_frame: int, num_frames: int):
        blink_len = int(self.maps[0].shape[0])
        if blink_len <= 0:
            return tuple(current_maps)
        idx = (torch.arange(int(num_frames), device=self.device) + int(start_frame)) % blink_len
        out = []
        for cur, blink_all, mask in zip(current_maps, self.maps, self.masks):
            blink = blink_all.index_select(0, idx).to(device=cur.device, dtype=cur.dtype)
            out.append(blink * mask.to(device=cur.device, dtype=cur.dtype) + cur * (1.0 - mask.to(device=cur.device, dtype=cur.dtype)))
        return tuple(out)


# --------------------------------------------------------------------------- video out
class Mp4Writer:
    """cv2.VideoWriter opened lazily on the first frame, so frames go to disk as they are
    rendered instead of piling up in VRAM."""

    def __init__(self, path: Path, fps: float):
        self.path = path
        self.fps = float(fps)
        self._w = None
        self.frames = 0

    def write_batch(self, rgb: np.ndarray) -> None:
        if self._w is None:
            h, w = rgb.shape[1:3]
            self._w = cv2.VideoWriter(str(self.path), cv2.VideoWriter_fourcc(*"mp4v"), self.fps, (w, h))
            if not self._w.isOpened():
                raise RuntimeError(f"cv2 could not open a video writer for {self.path}")
            print(f"[video] writing {w}x{h} @ {self.fps:g} fps -> {self.path}", flush=True)
        for frame in rgb:
            self._w.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
            self.frames += 1

    def close(self) -> None:
        if self._w is not None:
            self._w.release()
            self._w = None


def mux(silent: Path, audio: Path, out: Path, crf: int) -> bool:
    ffmpeg = shutil.which("ffmpeg")
    if not ffmpeg:
        print("[mux] ffmpeg not on PATH -- keeping the silent mp4v file", flush=True)
        return False
    rc = subprocess.run([
        ffmpeg, "-y", "-loglevel", "error",
        "-i", str(silent), "-i", str(audio),
        "-map", "0:v:0", "-map", "1:a:0",
        "-c:v", "libx264", "-crf", str(crf), "-preset", "medium", "-pix_fmt", "yuv420p",
        "-c:a", "aac", "-b:a", "192k", "-ac", "1",
        "-movflags", "+faststart", "-shortest",
        str(out),
    ]).returncode
    if rc != 0:
        print(f"[mux] ffmpeg failed (exit {rc}) -- keeping the silent mp4v file", flush=True)
        return False
    return True


# --------------------------------------------------------------------------- main
def parse_args() -> argparse.Namespace:
    ap = argparse.ArgumentParser(description="production speech2face avatar pipeline, offline")
    ap.add_argument("--imtalker_dir", required=True, help="path to <clone>/src/imtalker")
    ap.add_argument("--image", required=True)
    ap.add_argument("--audio", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--generator_path", required=True, help="weights/generator_last.ckpt")
    ap.add_argument("--renderer_path", required=True, help="weights/renderer.ckpt")
    ap.add_argument("--wav2vec_model_path", required=True, help="weights/wav2vec2-base-960h")
    ap.add_argument("--device", default="cuda")

    # --- the production motion settings (run_x86.sh)
    ap.add_argument("--fps", type=float, default=25.0)
    ap.add_argument("--wav2vec_sec", type=float, default=2.0, help="generator window; 2.0 in production")
    ap.add_argument("--fm_chunk_frames", type=int, default=2,
                    help="frames emitted per generator call. 2 = production (frozen head). "
                         "Setting this to wav2vec_sec*fps reverts to one-shot sampling and the "
                         "head starts moving again.")
    ap.add_argument("--num_prev_frames", type=int, default=10)
    ap.add_argument("--a_cfg_scale", type=float, default=1.24)
    ap.add_argument("--nfe", type=int, default=3)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--ode_method", default="euler")
    ap.add_argument("--shared_noise", action="store_true")
    ap.add_argument("--noise_seed", type=int, default=42)
    ap.add_argument("--noise_max_frames", type=int, default=5000)
    ap.add_argument("--noise_temporal_corr", type=float, default=0.0)
    ap.add_argument("--motion_prior_noise_blend", type=float, default=0.0)
    ap.add_argument("--motion_ref_blend", type=float, default=0.0,
                    help="blend motion toward the reference latent; >0 damps head tilt further")

    # --- wav2vec front end (replaces Helium + adapter_last.pt)
    ap.add_argument("--w2v_window_sec", type=float, default=20.0)
    ap.add_argument("--w2v_context_sec", type=float, default=2.0)

    # --- renderer / output
    ap.add_argument("--render_sub_batch", type=int, default=16)
    ap.add_argument("--renderer_precision", choices=("fp32", "fp16", "bf16"), default="fp32")
    ap.add_argument("--fp32", action="store_true")
    ap.add_argument("--tf32", action="store_true")
    ap.add_argument("--crf", type=int, default=18)
    ap.add_argument("--keep_silent", action="store_true")
    ap.add_argument("--save_motion", default="", help="optional .pt dump of the motion latents")

    # --- LoRA (production does not use it; _load_fm supports it)
    ap.add_argument("--lora_generator_path", default="")
    ap.add_argument("--lora_rank", type=int, default=0,
                    help="0 = read it from the checkpoint (hyper_parameters, else lora_A shape)")
    ap.add_argument("--lora_alpha", type=float, default=0.0,
                    help="0 = read it from the checkpoint, else 2*rank")
    ap.add_argument("--lora_dropout", type=float, default=0.05)
    ap.add_argument("--no_lora_pose_projection", action="store_true")
    ap.add_argument("--no_lora_audio_projection", action="store_true")
    ap.add_argument("--only_lora_pose_projection", action="store_true")

    # --- eye blink (off in production)
    ap.add_argument("--enable_eye_blink_composite", action="store_true")
    ap.add_argument("--blink_motion_path", default="")
    ap.add_argument("--eye_left_x", type=float, default=0.36)
    ap.add_argument("--eye_right_x", type=float, default=0.64)
    ap.add_argument("--eye_center_y", type=float, default=0.405)
    ap.add_argument("--eye_radius_x", type=float, default=0.145)
    ap.add_argument("--eye_radius_y", type=float, default=0.070)
    ap.add_argument("--eye_feather", type=float, default=0.10)
    return ap.parse_args()


def main() -> None:
    args = parse_args()
    imtalker_dir = Path(args.imtalker_dir).resolve()
    if not (imtalker_dir / "experiments" / "original_pod_8998" / "FM.py").is_file():
        raise SystemExit(
            f"--imtalker_dir has no experiments/original_pod_8998/FM.py: {imtalker_dir}\n"
            "That is the production generator this pipeline needs; the clone looks wrong."
        )
    for p in (args.image, args.audio, args.generator_path, args.renderer_path):
        if not Path(p).exists():
            raise SystemExit(f"missing input: {p}")
    if not (Path(args.wav2vec_model_path) / "config.json").is_file():
        raise SystemExit(f"--wav2vec_model_path has no config.json: {args.wav2vec_model_path}")

    add_imtalker_to_path(imtalker_dir)
    from generator.wav2vec2 import Wav2VecModel  # noqa: E402  (needs the sys.path above)

    if args.device.startswith("cuda") and not torch.cuda.is_available():
        raise SystemExit("--device cuda but torch.cuda.is_available() is False")
    if args.tf32:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    opt = build_opt(args)
    device = torch.device(args.device)
    dtype = {"fp32": torch.float32, "fp16": torch.float16, "bf16": torch.bfloat16}[args.renderer_precision]

    out = Path(args.out).resolve()
    out.parent.mkdir(parents=True, exist_ok=True)
    silent = out.with_name(out.stem + "_silent.mp4")

    t0 = time.time()
    fm = load_fm(opt, args, device)
    renderer = load_renderer(opt, device, dtype)

    # ---- reference image, encoded once (exactly as the live engine does at startup)
    ref_tensor = load_ref_image(Path(args.image), device, dtype)
    with torch.no_grad():
        f_r, g_r = renderer.dense_feature_encoder(ref_tensor)
        ref_x = renderer.latent_token_encoder(ref_tensor).to(dtype=torch.float32)
        m_r = renderer.latent_token_decoder(renderer.adapt(ref_x.to(dtype=dtype), g_r))
    print(f"[ref] {args.image} encoded, ref_x={tuple(ref_x.shape)}", flush=True)

    blink = None
    if opt.enable_eye_blink_composite:
        blink = BlinkCompositor(opt, renderer, g_r, device, dtype)

    # ---- audio -> 768-d Wav2Vec2 final hidden, 50 Hz -> 25 Hz
    wav16, audio_seconds = load_audio_16k(Path(args.audio))
    extractor = Wav2Vec2FeatureExtractor.from_pretrained(args.wav2vec_model_path, local_files_only=True)
    w2v = (
        Wav2VecModel.from_pretrained(
            args.wav2vec_model_path, local_files_only=True, attn_implementation="eager"
        )
        .to(device)
        .eval()
        .float()
    )
    t1 = time.time()
    feat_50 = wav2vec_50hz_long(
        wav16, extractor, w2v, device, args.w2v_window_sec, args.w2v_context_sec
    )
    total_frames = max(1, int(round(int(feat_50.shape[0]) / 2.0)))
    feat_25 = (
        F.interpolate(
            feat_50.T.unsqueeze(0), size=total_frames, mode="linear", align_corners=False
        )
        .squeeze(0)
        .T.contiguous()
    )
    print(
        f"[wav2vec] {tuple(feat_50.shape)} @50Hz -> {tuple(feat_25.shape)} @25Hz "
        f"in {time.time() - t1:.1f}s", flush=True
    )
    del feat_50
    # the generator's own wav2vec branch is not built (skip_fm_audio_encoder), so the
    # encoder is only needed for the features above -- free its VRAM before rendering
    del w2v
    if device.type == "cuda":
        torch.cuda.empty_cache()

    # ---- shared noise buffer, indexed by absolute frame (production: seed 42)
    noise_buf = None
    if args.shared_noise:
        n_noise = max(int(args.noise_max_frames), total_frames + int(args.fm_chunk_frames))
        gen = torch.Generator(device=device)
        gen.manual_seed(int(args.noise_seed))
        noise_buf = torch.randn(1, n_noise, int(opt.dim_w), device=device, generator=gen)
        print(f"[noise] shared buffer {tuple(noise_buf.shape)} seed={args.noise_seed}", flush=True)

    # ---- the streaming sample loop: this is what holds the head still
    step = max(1, int(args.fm_chunk_frames))
    window = int(float(opt.wav2vec_sec) * float(opt.fps))
    print(
        f"[sample] window={window}f emit={step}f/call prev={int(opt.num_prev_frames)}f "
        f"nfe={args.nfe} a_cfg_scale={args.a_cfg_scale} "
        f"-> {-(-total_frames // step)} calls for {total_frames} frames",
        flush=True,
    )

    writer = Mp4Writer(silent, opt.fps)
    stream_state = None
    abs_frame = 0
    render_cursor = 0
    pending: list[torch.Tensor] = []
    pending_n = 0
    motion_all: list[torch.Tensor] = []
    t_sample = t_render = 0.0

    @torch.no_grad()
    def render_pending() -> None:
        nonlocal pending, pending_n, render_cursor, t_render
        if not pending:
            return
        t_r0 = time.time()
        motion = torch.cat(pending, dim=0).to(device, dtype=dtype)
        n = int(motion.shape[0])
        g_r_sub = g_r.expand(n, -1)
        m_r_sub = tuple(m.expand(n, -1, -1, -1) for m in m_r)
        f_r_sub = [f.expand(n, -1, -1, -1) for f in f_r]
        m_c = renderer.latent_token_decoder(renderer.adapt(motion, g_r_sub))
        if blink is not None:
            m_c = blink.apply(m_c, render_cursor, n)
        frames = renderer.decode(m_c, m_r_sub, f_r_sub)
        frames_np = (
            frames.detach().float().clamp(0, 1).mul(255).to(torch.uint8)
            .permute(0, 2, 3, 1).contiguous().cpu().numpy()
        )
        writer.write_batch(frames_np)
        render_cursor += n
        pending = []
        pending_n = 0
        t_render += time.time() - t_r0

    with torch.no_grad():
        while abs_frame < total_frames:
            n = min(step, total_frames - abs_frame)
            a_feat = feat_25[abs_frame:abs_frame + n].unsqueeze(0).float()
            data: dict = {"a_feat": a_feat, "ref_x": ref_x}
            if noise_buf is not None:
                data["noise_init"] = noise_buf[:, abs_frame:abs_frame + n]

            t_s0 = time.time()
            motion, stream_state = fm.sample(
                data,
                a_cfg_scale=float(args.a_cfg_scale),
                nfe=int(args.nfe),
                stream_state=stream_state,
                return_state=True,
            )
            t_sample += time.time() - t_s0
            motion = motion.squeeze(0).detach()[:n]

            ref_blend = float(args.motion_ref_blend)
            if ref_blend > 0.0:
                ref_motion = ref_x.detach().float()
                if ref_motion.ndim == 3:
                    ref_motion = ref_motion[0]
                if ref_motion.ndim == 1:
                    ref_motion = ref_motion.unsqueeze(0)
                ref_motion = ref_motion[:1].expand(int(motion.shape[0]), -1).to(
                    device=motion.device, dtype=motion.dtype
                )
                blend = max(0.0, min(1.0, ref_blend))
                motion = motion.mul(1.0 - blend).add(ref_motion, alpha=blend)

            motion_all.append(motion.float().cpu())
            pending.append(motion)
            pending_n += int(motion.shape[0])
            abs_frame += n
            if pending_n >= int(args.render_sub_batch):
                render_pending()
            if abs_frame % 100 < n:
                print(
                    f"[progress] {abs_frame}/{total_frames} frames "
                    f"(sample {t_sample:.1f}s, render {t_render:.1f}s)",
                    flush=True,
                )
        render_pending()
    writer.close()
    if writer.frames == 0:
        raise SystemExit("the renderer produced no frames")

    # ---- head-motion report: the evidence that the head is actually frozen
    motion_cat = torch.cat(motion_all, dim=0)
    delta = (motion_cat[1:] - motion_cat[:-1]).abs() if motion_cat.shape[0] > 1 else torch.zeros(1, 1)
    stats = {
        "motion_frames": int(motion_cat.shape[0]),
        "latent_std_mean": round(float(motion_cat.std(dim=0).mean()), 5),
        "latent_std_max": round(float(motion_cat.std(dim=0).max()), 5),
        "frame_delta_mean": round(float(delta.mean()), 5),
        "frame_delta_p99": round(float(delta.flatten().quantile(0.99)), 5),
        "frame_delta_max": round(float(delta.max()), 5),
    }
    print(
        "[motion] per-dim std mean={latent_std_mean} max={latent_std_max} | "
        "frame-to-frame |delta| mean={frame_delta_mean} p99={frame_delta_p99} "
        "max={frame_delta_max}".format(**stats),
        flush=True,
    )
    if args.save_motion:
        mp = Path(args.save_motion)
        mp.parent.mkdir(parents=True, exist_ok=True)
        torch.save({"motion": motion_cat, "fps": float(opt.fps), "stats": stats}, mp)
        print(f"[motion] latents saved -> {mp}", flush=True)

    has_audio = mux(silent, Path(args.audio), out, args.crf)
    if not has_audio:
        shutil.move(str(silent), str(out))
    elif not args.keep_silent:
        silent.unlink(missing_ok=True)

    meta = {
        "output": str(out),
        "pipeline": "experiments.original_pod_8998.FM + renderer.models.IMTRenderer (production)",
        "image": str(Path(args.image).resolve()),
        "audio": str(Path(args.audio).resolve()),
        "generator_path": opt.generator_path,
        "renderer_path": opt.renderer_path,
        "lora_generator_path": opt.lora_generator_path,
        "frames": writer.frames,
        "fps": float(opt.fps),
        "video_seconds": round(writer.frames / float(opt.fps), 3),
        "audio_seconds": round(audio_seconds, 3),
        "has_audio_track": has_audio,
        "fm_chunk_frames": step,
        "generator_window_frames": window,
        "num_prev_frames": int(opt.num_prev_frames),
        "a_cfg_scale": float(args.a_cfg_scale),
        "nfe": int(args.nfe),
        "seed": int(args.seed),
        "shared_noise": bool(args.shared_noise),
        "noise_seed": int(args.noise_seed),
        "motion_ref_blend": float(args.motion_ref_blend),
        "noise_temporal_corr": float(args.noise_temporal_corr),
        "motion_prior_noise_blend": float(args.motion_prior_noise_blend),
        "eye_blink_composite": bool(opt.enable_eye_blink_composite),
        "renderer_precision": args.renderer_precision,
        "sample_seconds": round(t_sample, 2),
        "render_seconds": round(t_render, 2),
        "total_seconds": round(time.time() - t0, 2),
        "size_bytes": out.stat().st_size,
        "motion_stats": stats,
    }
    out.with_suffix(".json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(json.dumps(meta, indent=2), flush=True)


if __name__ == "__main__":
    main()


## 9 · Check the inputs

A quick look before spending GPU time. `rb_wav.wav` is 48 kHz mono, ~7.8 s; it is
resampled to the 16 kHz Wav2Vec2 wants, which gives ~390 frames at 50 Hz, ~195 at 25 Hz,
so ~7.8 s of video at 25 fps.

The preview is the reference frame exactly as the pipeline sees it: `rb.jpeg` resized to
512×512 with LANCZOS. There is no crop step — the live server does not crop either.


In [ ]:
from IPython.display import display, Image as IPyImage

IMAGE_PATH = resolve(IMAGE, REPO_DIR)
AUDIO_PATH = resolve(AUDIO, REPO_DIR)
OUT_PATH   = resolve(OUT, WORK)
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

for label, p in (("image", IMAGE_PATH), ("audio", AUDIO_PATH)):
    assert os.path.isfile(p), ("%s not found: %s\n"
        "Set IMAGE / AUDIO in section 1, or upload the file to /workspace." % (label, p))
    print("%-6s %s  (%.2f MB)" % (label, p, os.path.getsize(p) / 1e6))

sh('ffprobe -hide_banner -v error -show_entries '
   'stream=codec_name,sample_rate,channels,duration -of default=noprint_wrappers=1 "%s"'
   % AUDIO_PATH, check=False)

prep = textwrap.dedent("""
    import sys
    from pathlib import Path
    work, src, dst = sys.argv[1:4]
    sys.path.insert(0, work)
    from imtalker_pipeline_infer import load_ref_image
    import torch
    t = load_ref_image(Path(src), torch.device("cpu"), torch.float32)
    from PIL import Image
    import numpy as np
    arr = (t[0].permute(1, 2, 0).numpy() * 255).round().astype("uint8")
    Image.fromarray(arr).save(dst)
    print("preview ->", dst)
""")
Path(os.path.join(WORK, "_preview_ref.py")).write_text(prep, encoding="utf-8")
preview = os.path.join(WORK, "outputs", "reference_preview.png")
sh('"%s" _preview_ref.py "%s" "%s" "%s"' % (PY_ENV, WORK, IMAGE_PATH, preview))
display(IPyImage(filename=preview, width=320))


## 10 · Generate the video

This is the actual run. The streaming loop trades throughput for the frozen head: 2 frames
per generator call means ~98 generator calls for `rb_wav.wav` instead of 4, so expect
tens of seconds rather than a few. Rendering is batched at `RENDER_SUB_BATCH` frames
regardless, since the renderer is per-frame and stateless.

Watch three lines in the output:

* `[LoRA] targets=46 rank=16 alpha=32` — the static-head adapters are being injected.
* `[FM] ... matched N/M tensors` — if `matched` is near zero the checkpoint is not for
  this architecture and you would get a still, silent-looking face rather than an error.
  With the static-head checkpoint expect ~201 matched and ~211 unexpected: the unexpected
  ones are its `audio_encoder.wav2vec2.*` weights, which this path does not build because
  Wav2Vec2 runs separately (production passes `--skip_fm_audio_encoder` for the same reason).
* `[FM] LoRA overlay ... applied 92/92 tensors` — the EMA'd adapters landed.
* `[sample] window=50f emit=2f/call` — confirms the production loop is in use.
* `[motion] per-dim std ... frame-to-frame |delta| ...` — the head-motion report.


In [ ]:
cmd = (
    '"%s" imtalker_pipeline_infer.py'
    ' --imtalker_dir "%s"'
    ' --image "%s" --audio "%s" --out "%s"'
    ' --generator_path "%s" --renderer_path "%s" --wav2vec_model_path "%s"'
    ' --device cuda --fps %g'
    ' --wav2vec_sec %g --fm_chunk_frames %d --num_prev_frames %d'
    ' --a_cfg_scale %g --nfe %d --seed %d --noise_seed %d'
    ' --motion_ref_blend %g --noise_temporal_corr %g --motion_prior_noise_blend %g'
    ' --render_sub_batch %d --renderer_precision %s --crf %d'
    ' --save_motion "%s" --fp32 --tf32'
    % (PY_ENV, IMTALKER_DIR, IMAGE_PATH, AUDIO_PATH, OUT_PATH,
       GENERATOR_PATH, RENDERER_PATH, WAV2VEC_PATH,
       FPS, WAV2VEC_SEC, FM_CHUNK_FRAMES, NUM_PREV_FRAMES,
       A_CFG_SCALE, NFE, SEED, NOISE_SEED,
       MOTION_REF_BLEND, NOISE_TEMPORAL_CORR, MOTION_PRIOR_NOISE_BLEND,
       RENDER_SUB_BATCH, RENDERER_PRECISION, CRF,
       os.path.splitext(OUT_PATH)[0] + "_motion.pt")
)
if SHARED_NOISE:
    cmd += " --shared_noise"
if LORA_PATH:
    cmd += ' --lora_generator_path "%s"' % LORA_PATH
if ENABLE_EYE_BLINK:
    cmd += ' --enable_eye_blink_composite --blink_motion_path "%s"' % BLINK_PATH

t0 = time.time()
sh(cmd)
print("\nwall clock: %.1fs" % (time.time() - t0))

meta = json.loads(Path(OUT_PATH).with_suffix(".json").read_text())
print("\nvideo : %s  (%.1f MB)" % (meta["output"], meta["size_bytes"] / 1e6))
print("frames: %d at %g fps = %.2fs video / %.2fs audio"
      % (meta["frames"], meta["fps"], meta["video_seconds"], meta["audio_seconds"]))
print("motion: %s" % json.dumps(meta["motion_stats"]))
if not meta["has_audio_track"]:
    print("!! ffmpeg could not mux the audio -- the file is video-only")


## 11 · Play it, and read the head-motion report

The player embeds the MP4 as base64, which is fine for a few seconds of 512×512 and
unwieldy past ~30 MB; for anything long use the printed path.

The numbers under the player are what tell you the freeze worked:

| number | what it means |
|---|---|
| `latent_std_mean` / `latent_std_max` | spread of each motion-latent dimension over the whole clip. The mouth dimensions should move; if *every* dimension is near zero the face is not animating at all, which is a checkpoint problem, not a freeze |
| `frame_delta_mean` / `p99` / `max` | frame-to-frame change in the latent. A frozen head is a **small `max` relative to the `mean`** — no occasional large jump that would read as a head turn |

To see what the knob is actually doing, re-run section 10 with `FM_CHUNK_FRAMES = 50` in
section 1 and compare: same checkpoint, same audio, same seed, visibly moving head, and a
`frame_delta_max` several times larger.


In [ ]:
from IPython.display import HTML, display
import base64

size_mb = os.path.getsize(OUT_PATH) / 1e6
if size_mb <= 40:
    b64 = base64.b64encode(Path(OUT_PATH).read_bytes()).decode()
    display(HTML('<video width="512" controls autoplay loop '
                 'src="data:video/mp4;base64,%s"></video>' % b64))
else:
    print("%.0f MB is too big to embed -- download it instead" % size_mb)

print("\nfile      :", OUT_PATH)
print("motion .pt:", os.path.splitext(OUT_PATH)[0] + "_motion.pt")
print("runpodctl : runpodctl send %s" % OUT_PATH)
print("\nhead-motion report:")
for k, v in meta["motion_stats"].items():
    print("  %-18s %s" % (k, v))
print("\nfull summary:")
print(json.dumps(meta, indent=2))


## 12 · The freeze knob, the other knobs, and troubleshooting

### What actually freezes the head

Two things, and you want both.

**1. The static-head LoRA.** `lora_exp4_lookahead_120ms_static_head_last.ckpt` is a
rank-16 adapter trained for 20 000 steps on top of `exp4_lookahead_120ms` with the base
frozen, and its `exp_name` says what it is for. The adapters sit on all 8 FMT blocks'
attention, MLP and adaLN modulation, on the two `fmt.decoder` linears, and on the `pose` /
`cam` / `gaze` / `audio` projections — so they can damp the pose conditioning directly.
`STATIC_HEAD_LORA` turns them on and off.

**2. The streaming sample loop.** Upstream IMTalker samples the whole clip in one pass:
initial noise drawn fresh. Nothing anchors the trajectory, so the low-frequency component
of the motion latent wanders and you see the head drift and tilt.

Your pipeline runs the same generator as a **stream**:

| | upstream `generate.py` | this pipeline (`run_x86.sh`) |
|---|---|---|
| generator window | 50 frames | 50 frames |
| frames kept per call | 50 (all of them) | **2** |
| conditioning on previous motion | none | `prev_sample`, last 10 frames |
| initial noise | fresh `randn` per chunk | 25 frames carried from the previous call's `x0`, rest from a shared buffer indexed by absolute frame |
| `a_cfg_scale` / `nfe` | 2.0 / 10 | 1.24 / 3 |

Each call is pinned to where the face already was by 10 frames of real previous motion and
25 frames of the previous initial noise, and it only has to produce 80 ms. There is no room
for the pose to go anywhere, while the 2 fresh audio frames still drive the mouth. The
shared noise buffer removes the last source of drift: the same absolute frame always gets
the same noise, so re-running is bit-for-bit reproducible.

`FM_CHUNK_FRAMES` is that knob:

| value | behaviour |
|---|---|
| `2` | production. Head frozen. ~98 generator calls for 8 s of audio |
| `4`–`8` | still essentially frozen, 2–4× faster |
| `12`–`25` | the head starts to move a little |
| `50` (= `WAV2VEC_SEC × FPS`) | one-shot sampling — upstream IMTalker's behaviour, head moves |

If you want it stiller still, these three are already wired and all `0.0` in production:

| knob | effect |
|---|---|
| `MOTION_REF_BLEND` | blends every motion latent toward the reference image's latent. `0.05`–`0.15` kills residual tilt; too high and the mouth closes up as well |
| `NOISE_TEMPORAL_CORR` | AR(1) smoothing on the initial noise, `0.0`–`0.999`. Smooths everything, mouth included |
| `MOTION_PRIOR_NOISE_BLEND` | blends the previous motion's linear extrapolation into the initial noise |

### The other knobs

| knob | default | what moving it does |
|---|---|---|
| `STATIC_HEAD_LORA` | `True` | apply the `lora_exp4_lookahead_120ms_static_head` adapters. `False` runs the base generator alone, which is literally what `run_x86.sh` does — useful for telling apart how much of the freeze is the checkpoint and how much is the streaming loop |
| `A_CFG_SCALE` | `1.24` | audio classifier-free guidance. The production value. Upstream's `2.0`–`3.0` articulates harder but also re-excites head motion |
| `NFE` | `3` | ODE steps. Production runs 3 because it has a 40 ms budget; offline you can afford `5`–`10`, which slightly cleans up the mouth. It changes the trajectory, so the result is not identical to live |
| `SEED` / `NOISE_SEED` | `42` | same seeds + same audio + same image = the same video, bit for bit |
| `WAV2VEC_SEC` | `2.0` | **checkpoint-coupled — leave it.** It sets the generator's window to 50 frames, which is the `wav2vec_sec = 2` recorded in the static-head checkpoint's own `hyper_parameters` |
| `FPS` | `25.0` | **leave it.** The generator emits 25 fps by construction |
| `ENABLE_EYE_BLINK` | `False` | production has blinking off. `True` composites `blink_motion.pt` into the eye region — a frozen head with blinking eyes, which usually reads better over a long clip |
| `RENDER_SUB_BATCH` | `16` | frames per renderer call. Purely speed vs VRAM; it does not change a pixel |

### Other inputs

```python
IMAGE = "/workspace/my_photo.jpg"    # resized to 512x512, no crop -- frame it yourself
AUDIO = "/workspace/my_speech.mp3"   # torchaudio first, librosa fallback
OUT   = "outputs/my_avatar.mp4"
```

The reference photo is **not** cropped or face-detected, exactly as in production, so a
wide snapshot will render as a wide snapshot. Crop it to a square head-and-shoulders
portrait first — `rb.jpeg` already is one.

### Your own checkpoints

If you have `generator_last.ckpt` on the pod, put its directory in `WEIGHTS_DIR` (or just
leave `WEIGHTS_DIR = ""` — `/workspace/speech2face/weights` is searched first). Section 7
then uses it as the base and applies the static-head adapters on top, which is the pairing
they were trained for. `STATIC_HEAD_LORA = False` reproduces `run_x86.sh` exactly: your
base, no adapters.

`GENERATOR_OVERRIDE` and `LORA_OVERRIDE` take any pair of paths. The loader accepts a raw
`state_dict`, a Lightning `{"state_dict": ...}`, a nested `{"state_dict": {"model": ...}}`,
an `ema_state_dict`, and the `model.` / `gen.` prefixes the training scripts write; it skips
tensors whose shape does not match and prints the counts, so read those lines.

### A whole folder of audio against one image

```python
for wav in sorted(Path("/workspace/clips").glob("*.wav")):
    out = os.path.join(WORK, "outputs", wav.stem + ".mp4")
    sh(cmd.replace(AUDIO_PATH, str(wav)).replace(OUT_PATH, out))
```

Each call reloads the checkpoints (~15 s). For many clips, import
`imtalker_pipeline_infer` and reuse one loaded generator/renderer pair.

### Troubleshooting

| symptom | what to do |
|---|---|
| `matched 0/N tensors` for the generator | the checkpoint is not a production IMTalker generator, or the copy is truncated. The static-head file should be 558.0 MB |
| `the LoRA adapters matched nothing` | the rank is wrong for this checkpoint. Leave `--lora_rank` at `0` so the script reads it from `hyper_parameters` (16 here) instead of assuming `_load_fm`'s default of 64 |
| `size mismatch for ...lora_A.weight` | same cause, caught by torch first. Drop any `--lora_rank` override |
| `unexpected=211` on the generator load | expected and harmless: those are the checkpoint's `audio_encoder.wav2vec2.*` weights, which this path does not build because Wav2Vec2 runs separately |
| the head still moves | `STATIC_HEAD_LORA` is `False`, or `FM_CHUNK_FRAMES` is not 2, or `SHARED_NOISE` is off, or `A_CFG_SCALE` was raised. Check the `[LoRA] targets=46` and `[sample] window=.. emit=..` lines in section 10 |
| the face is completely still, mouth included | the generator matched few tensors, or the audio decoded to silence. Check `ffprobe` in section 9 and `latent_std_max` in the report — a working clip has some dimensions with visible spread |
| used the fallback `renderer.ckpt` | upstream's 2121.1 MB renderer is the same file your pipeline ships, so this is fine. Point `WEIGHTS_DIR` at yours if you have replaced it with a fine-tune |
| `CUDA out of memory` | lower `RENDER_SUB_BATCH` to `4` or `2`. Nothing else accumulates on the GPU. Long audio also windows Wav2Vec2 automatically (`--w2v_window_sec`, 20 s) |
| `numpy 2.x ... _ARRAY_API not found` | re-run section 6; its step 7/7 pins numpy back to 1.26.4 |
| `local_files_only=True` error on wav2vec | the `wav2vec2-base-960h/` directory is incomplete. Delete it from `checkpoints/` and re-run section 7 |
| `ffmpeg not on PATH` | re-run section 3. Without it you still get the video, just silent |
| pod restarted, nothing works | re-run sections 1 → 7. Everything is on `/workspace`, so 6 is a no-op and you are back in ~1 min |
